# AELIONIX BLACKFORGE — Phase 1 Colab Validation

This notebook performs a deterministic, one-click validation of the Blackforge Phase 1 runtime.

**What this validates:**
- Repository integrity and commit verification
- Dependency installation (torch, transformers, accelerate)
- Hardware detection (GPU/CPU)
- All Blackforge imports
- Full automated test suite (177 tests)
- Bootstrap and health verification
- Real HuggingFace model inference through the provider abstraction
- Structured output generation
- Tool-call normalization
- ModelRouter validation

**What this does NOT do:**
- No offensive security actions
- No reconnaissance or scanning
- No credential attacks
- No autonomous attack planning

**Runtime:** Google Colab (CPU or GPU). GPU recommended for real inference test.

---
## 1. Runtime Information

In [ ]:
import sys
import platform

print("Blackforge Phase 1 Colab Validation")
print("=" * 60)
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("=" * 60)

assert sys.version_info >= (3, 10), f"Blackforge requires Python 3.10+, got {sys.version}"
print("Python version check: PASS")

---
## 2. GPU / Hardware Verification

In [ ]:
print("CUDA / hardware verification")
print("=" * 60)

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print(
            "VRAM:",
            round(
                torch.cuda.get_device_properties(0).total_memory / 1024**3,
                2,
            ),
            "GB",
        )
        _hw_device = "cuda"
    else:
        print("GPU unavailable — CPU mode")
        _hw_device = "cpu"
except ImportError:
    print("PyTorch not installed yet; installation happens in the next stage.")
    _hw_device = "unknown"

print("=" * 60)

---
## 3. Repository Acquisition

In [ ]:
from pathlib import Path
import subprocess

# ── Configuration (edit here if fork changes) ──────────────────────────
REPO_URL = "https://github.com/Sagelord00000001/Blackforge.git"
REPO_DIR = Path("/content/blackforge")
# ───────────────────────────────────────────────────────────────────────

if REPO_DIR.exists() and (REPO_DIR / "blackforge" / "__init__.py").exists():
    print(f"Repository already exists at {REPO_DIR}, updating...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    if REPO_DIR.exists():
        import shutil
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True,
    )

import os
os.chdir(str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")

---
## 4. Commit Verification

In [ ]:
import subprocess

EXPECTED_PHASE1_COMMIT = "1e54de4"

result = subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h"],
    capture_output=True, text=True, check=True,
)
current_commit = result.stdout.strip()

print(f"Expected Phase 1 baseline: {EXPECTED_PHASE1_COMMIT}")
print(f"Current repository commit: {current_commit}")

# Check if current commit is at or after Phase 1
result_log = subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "--oneline"],
    capture_output=True, text=True, check=True,
)
commits = [line.split()[0] for line in result_log.stdout.strip().splitlines()]

if EXPECTED_PHASE1_COMMIT in commits:
    print("Commit verification: PASS (Phase 1 baseline found)")
elif any(c.startswith(EXPECTED_PHASE1_COMMIT[:4]) for c in commits):
    print("Commit verification: PASS (Phase 1 baseline found, abbreviated match)")
else:
    # Check if Phase 1 commit is an ancestor (repo advanced past it)
    result_merge = subprocess.run(
        ["git", "-C", str(REPO_DIR), "merge-base", "--is-ancestor",
         EXPECTED_PHASE1_COMMIT, current_commit],
        capture_output=True, check=False,
    )
    if result_merge.returncode == 0:
        print("Commit verification: PASS (repo advanced past Phase 1)")
    else:
        print(f"WARNING: Phase 1 commit {EXPECTED_PHASE1_COMMIT} not found in history.")
        print("The repo may predate Phase 1. Continuing anyway.")

---
## 5. Install Blackforge

In [ ]:
# Install build backend first, then Blackforge with all extras
!pip install hatchling --quiet
!pip install -e ".[colab,dev,llm]"

# Verify installation succeeded
import blackforge
print("Blackforge import: PASS")

---
## 6. Environment / Import Health Check

In [ ]:
import importlib

modules = [
    "blackforge",
    "blackforge.core.config",
    "blackforge.core.errors",
    "blackforge.core.types",
    "blackforge.core.logging",
    "blackforge.runtime.bootstrap",
    "blackforge.runtime.hardware",
    "blackforge.intelligence.llm",
    "blackforge.intelligence.llm.base",
    "blackforge.intelligence.llm.huggingface",
    "blackforge.intelligence.llm.loader",
    "blackforge.intelligence.llm.ollama",
    "blackforge.intelligence.llm.mock",
    "blackforge.intelligence.context",
    "blackforge.intelligence.tokens",
    "blackforge.intelligence.structured",
    "blackforge.intelligence.routing.router",
    "blackforge.authorization",
    "blackforge.capabilities.registry",
    "blackforge.evidence.store",
    "blackforge.memory.base",
    "blackforge.memory.models",
    "blackforge.mission.manager",
    "blackforge.scope.validator",
]

_import_failures = []
for module in modules:
    try:
        importlib.import_module(module)
    except Exception as e:
        _import_failures.append((module, str(e)))

if _import_failures:
    for mod, err in _import_failures:
        print(f"  FAIL: {mod} — {err}")
    raise RuntimeError(f"Import health check failed: {len(_import_failures)} module(s)")

print(f"Blackforge backend imports OK ({len(modules)} modules verified).")

---
## 7. Automated Regression Tests

In [ ]:
import subprocess
import sys

print("Running automated test suite...")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--tb=short"],
    capture_output=True, text=True, cwd=str(REPO_DIR),
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:] if result.stderr else "")
    raise RuntimeError(f"pytest failed with exit code {result.returncode}")

print("Automated test suite: PASS")

---
## 8. Bootstrap Verification

In [ ]:
from blackforge.runtime.bootstrap import bootstrap

app = bootstrap()
assert app.healthy(), "Blackforge health check failed"
verification = app.verify()
assert all(verification.values()), (
    f"Blackforge verification failed: {verification}"
)

for k, v in verification.items():
    symbol = "PASS" if v else "FAIL"
    print(f"  [{symbol}] {k}")

print("\nBlackforge bootstrap: PASS")

---
## 9. Provider Configuration

In [ ]:
from blackforge.core.config import load_config

config = load_config()
llm = config.llm

print("Provider configuration")
print("=" * 60)
print(f"Provider:       {llm.provider}")
print(f"Model:          {llm.model}")
print(f"Device:         {llm.device}")
print(f"Dtype:          {llm.dtype}")
print(f"Context length: {llm.context_length}")
print(f"Max tokens:     {llm.max_output_tokens}")
print(f"Temperature:    {llm.temperature}")
print(f"Quantization:   {llm.quantization}")
print(f"Allow download: {llm.allow_download}")
print(f"Cache dir:      {llm.cache_dir}")
print("=" * 60)

---
## 10–12. Real Hugging Face Inference Test

This is the most important test. It validates the full provider abstraction chain:

```
Blackforge → ModelRouter → LLMProvider → HuggingFaceProvider → ModelLoader → Model → Inference
```

In [ ]:
import os
import time

# Set the provider to huggingface for this test
os.environ["BLACKFORGE_LLM_PROVIDER"] = "huggingface"

from blackforge.core.config import BlackforgeConfig, LLMConfig
from blackforge.intelligence.llm.huggingface import HuggingFaceProvider
from blackforge.intelligence.llm.base import LLMRequest

hf_config = LLMConfig(
    provider="huggingface",
    model="Qwen/Qwen2.5-3B-Instruct",
    device="auto",
    dtype="auto",
    context_length=2048,
    max_output_tokens=128,
    temperature=0.7,
)

print("Loading real Hugging Face model...")
print("This may take several minutes on a fresh Colab runtime.")
print("=" * 60)

provider = HuggingFaceProvider(hf_config)

# Verify torch/transformers are importable
assert provider.health_check(), (
    "Health check failed — torch or transformers not installed. "
    "Ensure you ran: pip install -e '.[llm]'"
)
print("torch/transformers importable: PASS")

# Run real inference
start = time.time()
result = provider.verify_inference(
    prompt=(
        "You are validating an AI runtime. "
        "Return exactly one short sentence confirming "
        "that Blackforge runtime inference is operational."
    )
)
elapsed = time.time() - start

print("=" * 60)
print(f"Real inference:      {'PASS' if result['success'] else 'FAIL'}")
print(f"Provider:            {result['provider']}")
print(f"Model:               {result['model']}")
print(f"Device:              {result['device']}")
print(f"Dtype:               {result.get('dtype')}")
print(f"Elapsed:             {result['latency_seconds']} seconds")
print(f"Usage:               {result.get('usage', {})}")
print(f"Response:\n{result['response']}")
print("=" * 60)

assert result["success"], (
    f"Real inference failed. Model did not produce output. "
    f"Provider: {result['provider']}, Model: {result['model']}"
)

# Verify no mock provider was used
assert result["provider"] != "mock", (
    "Mock provider was used instead of real HuggingFace provider."
)

_real_inference_result = result
print("\nReal HuggingFace inference: PASS")

---
## 13. Structured Output Test

In [ ]:
from blackforge.intelligence.llm.base import LLMRequest
from blackforge.intelligence.structured import parse_structured_response

schema = {
    "type": "object",
    "properties": {
        "status": {"type": "string"},
        "message": {"type": "string"},
    },
    "required": ["status", "message"],
}

print("Testing structured generation through provider abstraction...")

structured_req = LLMRequest(
    prompt="Return a JSON object with 'status' set to 'ok' and 'message' set to 'runtime operational'.",
    max_tokens=128,
    temperature=0.1,
)

raw_response = provider.structured_generate(structured_req, schema=schema)

print(f"Raw response content: {raw_response.content}")

parsed = raw_response.raw.get("parsed_structured")
if parsed is None:
    # Fallback: parse the raw content ourselves
    output = parse_structured_response(
        raw_response.content or "",
        schema=schema,
        model_used=hf_config.model,
    )
    parsed = output.parsed

print(f"Parsed result: {parsed}")

assert isinstance(parsed, dict), f"Expected dict, got {type(parsed)}"
assert "status" in parsed, f"Missing 'status' key in: {parsed}"
assert "message" in parsed, f"Missing 'message' key in: {parsed}"

print("Structured output: PASS")

---
## 14. Tool-Call Normalization Test

In [ ]:
from blackforge.intelligence.llm.base import LLMRequest, LLMResponse, ToolCall, Usage
from blackforge.intelligence.llm.mock import MockLLMProvider
from blackforge.capabilities.registry import CapabilityRegistry
from blackforge.capabilities.interface import CapabilityResult
from blackforge.intelligence.context import ChatContext


class MockToolCallProvider(MockLLMProvider):
    """Mock that returns a normalized tool call."""
    def generate(self, request: LLMRequest) -> LLMResponse:
        if request.tools:
            return LLMResponse(
                content=None,
                tool_calls=[ToolCall(name="mock_discovery", arguments={"target": "example.com"}, id="call_001")],
                model=self._model,
                provider="mock",
                usage=Usage(prompt_tokens=10, completion_tokens=5, total_tokens=15),
                finish_reason="tool_calls",
            )
        return super().generate(request)


print("Testing tool-call normalization flow...")

# Step 1: LLM proposes a tool call
llm = MockToolCallProvider("mock")
registry = CapabilityRegistry()
from blackforge.capabilities.mock import MockDiscoveryCapability
registry.register(MockDiscoveryCapability())

request = LLMRequest(
    prompt="scan example.com",
    tools=[{"name": "mock_discovery", "parameters": {"target": {"type": "string"}}}],
)
llm_resp = llm.tool_call(request, tools=[{"name": "mock_discovery"}])

assert llm_resp.tool_calls is not None, "No tool calls returned"
assert len(llm_resp.tool_calls) == 1, f"Expected 1 tool call, got {len(llm_resp.tool_calls)}"

tc = llm_resp.tool_calls[0]
assert tc.name == "mock_discovery", f"Wrong tool name: {tc.name}"
print(f"  Tool call: {tc.name}({tc.arguments})")

# Step 2: Resolve capability
assert registry.has(tc.name)
cap = registry.get(tc.name)

# Step 3: Execute capability
cap_result: CapabilityResult = cap.execute(target=tc.arguments["target"])
assert cap_result.success is True
print(f"  Capability result: {cap_result.output}")

# Step 4: Feed back into context
ctx = ChatContext(system_prompt="You are a security analyst.")
ctx.add_user("scan example.com")
ctx.add_assistant("calling tool", tool_calls=[tc])
ctx.add_tool_result(tool_call_id=tc.id or "", name=tc.name, output=str(cap_result.output))

messages = ctx.to_dict()
assert len(messages) == 4, f"Expected 4 messages (system+user+assistant+tool), got {len(messages)}"
assert messages[0]["role"] == "system"
assert messages[1]["role"] == "user"
assert messages[2]["role"] == "assistant"
assert messages[3]["role"] == "tool"

print("Tool-call normalization: PASS")

---
## 15. Model Router Test

In [ ]:
from blackforge.core.types import TaskCategory
from blackforge.intelligence.llm.base import LLMRequest
from blackforge.intelligence.llm.mock import MockLLMProvider
from blackforge.intelligence.routing.router import ModelRouter, RoutingRule

print("Testing ModelRouter...")
print("=" * 60)

default_provider = MockLLMProvider("default_model")
router = ModelRouter(default_provider=default_provider)

# Test all TaskCategory values
categories_tested = []
for cat in TaskCategory:
    resp = router.route(cat, LLMRequest(prompt=f"test {cat.value}"))
    assert resp.model == "default_model", f"Router failed for {cat.value}: got {resp.model}"
    categories_tested.append(cat.value)
    print(f"  [{cat.value}] routed to default -> PASS")

# Test with custom rule
specialized = MockLLMProvider("specialized_model")
router.register_provider("specialized", specialized)
router.add_rule(RoutingRule(category=TaskCategory.ANALYSIS, provider_name="specialized", priority=10))

resp = router.route(TaskCategory.ANALYSIS, LLMRequest(prompt="analyze"))
assert resp.model == "specialized_model", f"Custom rule failed: got {resp.model}"
print(f"  [analysis] routed to specialized -> PASS")

# Health check
health = router.health_check()
print(f"\nRouter health: {health}")
assert all(health.values()), f"Router health check failed: {health}"

print("=" * 60)
print(f"Model router: PASS ({len(categories_tested)} categories + custom rule)")

---
## 16. Runtime Diagnostics

In [ ]:
from blackforge.runtime.hardware import detect_hardware

hw = detect_hardware()
hw_dict = hw.to_dict()

print("Runtime diagnostics")
print("=" * 60)
print(f"Provider:            {_real_inference_result['provider']}")
print(f"Model:               {_real_inference_result['model']}")
print(f"Device:              {_real_inference_result['device']}")
print(f"Dtype:               {_real_inference_result.get('dtype')}")
print(f"Inference time:      {_real_inference_result['latency_seconds']}s")
usage = _real_inference_result.get('usage', {})
print(f"Input tokens:        {usage.get('prompt_tokens', 'N/A')}")
print(f"Output tokens:       {usage.get('completion_tokens', 'N/A')}")
print(f"Total tokens:        {usage.get('total_tokens', 'N/A')}")
print(f"Retry count:         {usage.get('retry_count', 0)}")
print(f"Hardware detected:   {hw_dict['device']}")
print(f"CUDA available:      {hw_dict['cuda_available']}")
print(f"GPU name:            {hw_dict['gpu_name']}")
print(f"GPU memory:          {hw_dict['gpu_memory_gb']} GB")
print(f"CPU count:           {hw_dict['cpu_count']}")
print(f"MPS available:       {hw_dict['has_mps']}")
print("=" * 60)
print("(No API keys, tokens, or secrets printed above.)")

---
## 17. PASS/FAIL Summary

In [ ]:
# Re-run all checks and compile results
RESULTS = {}

# 1. Repository
RESULTS["repository"] = (REPO_DIR / "blackforge" / "__init__.py").exists()

# 2. Imports
RESULTS["imports"] = len(_import_failures) == 0

# 3. Tests (already verified above)
RESULTS["tests"] = True  # Would have raised if failed

# 4. Bootstrap
RESULTS["bootstrap"] = app.healthy()

# 5. Hardware
RESULTS["hardware"] = True  # Always true — CPU or GPU

# 6. Real inference
RESULTS["real_inference"] = (
    _real_inference_result["success"]
    and _real_inference_result["provider"] != "mock"
)

# 7. Structured output
RESULTS["structured_output"] = (
    parsed is not None
    and isinstance(parsed, dict)
    and "status" in parsed
)

# 8. Tool-call flow
RESULTS["tool_call_normalization"] = True  # Verified above

# 9. Router
RESULTS["router"] = all(health.values())

# Print summary
print("=" * 60)
print("BLACKFORGE PHASE 1 VALIDATION")
print("=" * 60)

labels = {
    "repository": "Repository",
    "imports": "Imports",
    "tests": "Automated tests",
    "bootstrap": "Bootstrap",
    "hardware": "Hardware",
    "real_inference": "Real inference",
    "structured_output": "Structured output",
    "tool_call_normalization": "Tool-call flow",
    "router": "Model router",
}

for key in labels:
    status = "PASS" if RESULTS[key] else "FAIL"
    print(f"{labels[key]:<25} {status}")

print("=" * 60)

if all(RESULTS.values()):
    print("OVERALL RESULT: PASS")
else:
    failed = [labels[k] for k, v in RESULTS.items() if not v]
    print(f"OVERALL RESULT: FAIL — {', '.join(failed)}")

print("=" * 60)

In [ ]:
# Clean up: close provider to release memory
provider.close()
print("Provider closed. Validation complete.")